# COVID-19 Mortality: Pre-processing and Training Data Development

This notebook prepares the final analytical dataset for predictive modeling. Building on the data wrangling and exploratory data analysis phases, the dataset will be refined and transformed into a modeling-ready format.

The primary objective of this phase is to create a training dataset that can be used to investigate country-level factors associated with COVID-19 mortality.

Specific tasks include:

- Selecting variables for modeling
- Evaluating and addressing missing values
- Creating dummy variables for categorical features
- Applying feature scaling where appropriate
- Assessing potential variable transformations
- Splitting the data into training and testing subsets

The resulting dataset will serve as the foundation for predictive modeling in subsequent project phases.

In [1]:
packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "sklearn",
    "scipy",
    "statsmodels"
]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg}")

✓ numpy
✓ pandas
✓ matplotlib
✓ seaborn
✓ sklearn
✓ scipy
✓ statsmodels


In [2]:
!which python
!which pip

/Users/papa/miniforge3/envs/ds/bin/python
/Users/papa/miniforge3/envs/ds/bin/pip


In [3]:
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import statsmodels
import scipy

import sys

pd.set_option("display.max_columns", None)

print(sys.executable)

/Users/papa/miniforge3/envs/ds/bin/python


In [4]:
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("statsmodels:", statsmodels.__version__)

numpy: 2.0.1
pandas: 2.3.3
scipy: 1.16.0
sklearn: 1.7.1
statsmodels: 0.14.6


In [5]:
covid = pd.read_csv(
    "../data/processed/covid_analysis_dataset.csv"
)

covid.head()

,country,code,continent,population,total_cases,total_cases_per_million,total_deaths,total_deaths_per_million,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,stringency_index,median_age,gdp_per_capita,hospital_beds_per_thousand,life_expectancy,diabetes_prevalence,iso_code,country_wb,population_density,health_expenditure_per_capita,health_expenditure_pct_gdp
0,Afghanistan,AFG,Asia,40578847.0,235214.0,5796.4683,7998.0,197.09776,47.195450,45.270844,6.727495,27.394580,16.752001,1983.812622,0.35,65.616997,11.7,AFG,Afghanistan,61.328691,81.521126,21.508444
1,Albania,ALB,Europe,2827614.0,337234.0,119264.5100,3608.0,1275.98740,47.717087,45.244260,14.230054,41.782108,35.943001,21641.074219,2.90,78.768799,10.6,ALB,Albania,90.867226,465.570435,7.357504
2,Algeria,DZA,Africa,45477391.0,272440.0,5990.6690,6881.0,151.30595,17.239624,14.251447,1.265796,48.367728,27.983000,15501.919922,1.61,76.128899,17.5,DZA,Algeria,18.793445,208.939117,5.021889
3,American Samoa,ASM,Oceania,48365.0,8359.0,172831.6000,34.0,702.98770,NaN,NaN,NaN,NaN,27.927000,NaN,NaN,72.752098,NaN,ASM,American Samoa,246.125000,NaN,NaN
4,Andorra,AND,Europe,79722.0,48015.0,602280.4400,159.0,1994.43070,72.643684,67.109460,54.026493,33.486770,42.832001,65928.304688,NaN,84.016403,10.1,AND,Andorra,166.731915,3668.447510,8.646717


## Variable Selection

Based on the exploratory analysis, a subset of variables was selected for potential inclusion in predictive models. These variables represent demographic, healthcare, economic, epidemiological, and policy-related factors that may influence COVID-19 mortality.

In [6]:
target = "total_deaths_per_million"

candidate_vars = [
    "median_age",
    "life_expectancy",
    "total_cases_per_million",
    "gdp_per_capita",
    "hospital_beds_per_thousand",
    "people_fully_vaccinated_per_hundred",
    "stringency_index",
    "continent"
]

model_data = covid[
    [target] + candidate_vars
].copy()

model_data.head()


,total_deaths_per_million,median_age,life_expectancy,total_cases_per_million,gdp_per_capita,hospital_beds_per_thousand,people_fully_vaccinated_per_hundred,stringency_index,continent
0,197.09776,16.752001,65.616997,5796.4683,1983.812622,0.35,45.270844,27.394580,Asia
1,1275.98740,35.943001,78.768799,119264.5100,21641.074219,2.90,45.244260,41.782108,Europe
2,151.30595,27.983000,76.128899,5990.6690,15501.919922,1.61,14.251447,48.367728,Africa
3,702.98770,27.927000,72.752098,172831.6000,NaN,NaN,NaN,NaN,Oceania
4,1994.43070,42.832001,84.016403,602280.4400,65928.304688,NaN,67.109460,33.486770,Europe


In [7]:
model_data.isna().sum()

total_deaths_per_million                6
median_age                              2
life_expectancy                         2
total_cases_per_million                 6
gdp_per_capita                         40
hospital_beds_per_thousand             71
people_fully_vaccinated_per_hundred    24
stringency_index                       54
continent                               0
dtype: int64

In [8]:
model_data.shape

(239, 9)

In [9]:
model_data.dropna().shape

(151, 9)

## Missing Data Strategy

Removing all observations containing missing values would reduce the dataset from 239 countries to 151 countries, resulting in the loss of approximately 37% of available observations.

To preserve sample size while retaining important predictors identified during exploratory analysis, missing numerical values will be imputed using median values. Median imputation was selected because several variables exhibit skewed distributions and are therefore less appropriately represented by mean values.

In [10]:

numeric_cols = [
    "median_age",
    "life_expectancy",
    "total_cases_per_million",
    "gdp_per_capita",
    "hospital_beds_per_thousand",
    "people_fully_vaccinated_per_hundred",
    "stringency_index"
]


In [11]:
imputer = SimpleImputer(strategy="median")

model_data[numeric_cols] = imputer.fit_transform(
    model_data[numeric_cols]
)

In [12]:
model_data.isna().sum()

total_deaths_per_million               6
median_age                             0
life_expectancy                        0
total_cases_per_million                0
gdp_per_capita                         0
hospital_beds_per_thousand             0
people_fully_vaccinated_per_hundred    0
stringency_index                       0
continent                              0
dtype: int64

In [13]:
model_data = model_data.dropna(
    subset=["total_deaths_per_million"]
)


## Creating Dummy Variables

The `continent` variable is categorical and cannot be used directly in most machine learning models. To include continent as a predictor, it will be converted into dummy variables using one-hot encoding.

One category will be dropped to avoid perfect multicollinearity among the dummy variables.

In [14]:
model_data_encoded = pd.get_dummies(
    model_data,
    columns=["continent"],
    drop_first=True
)

model_data_encoded.head()

,total_deaths_per_million,median_age,life_expectancy,total_cases_per_million,gdp_per_capita,hospital_beds_per_thousand,people_fully_vaccinated_per_hundred,stringency_index,continent_Asia,continent_Europe,continent_North America,continent_Oceania,continent_South America
0,197.09776,16.752001,65.616997,5796.4683,1983.812622,0.35,45.270844,27.394580,True,False,False,False,False
1,1275.98740,35.943001,78.768799,119264.5100,21641.074219,2.90,45.244260,41.782108,False,True,False,False,False
2,151.30595,27.983000,76.128899,5990.6690,15501.919922,1.61,14.251447,48.367728,False,False,False,False,False
3,702.98770,27.927000,72.752098,172831.6000,18068.505859,2.38,62.440320,43.273038,False,False,False,True,False
4,1994.43070,42.832001,84.016403,602280.4400,65928.304688,2.38,67.109460,33.486770,False,True,False,False,False


In [15]:
model_data_encoded.columns.tolist()

['total_deaths_per_million',
 'median_age',
 'life_expectancy',
 'total_cases_per_million',
 'gdp_per_capita',
 'hospital_beds_per_thousand',
 'people_fully_vaccinated_per_hundred',
 'stringency_index',
 'continent_Asia',
 'continent_Europe',
 'continent_North America',
 'continent_Oceania',
 'continent_South America']

In [16]:
model_data_encoded.shape

(233, 13)

In [ ]:
covid["log_total_deaths_per_million"] = np.log1p(
    covid["total_deaths_per_million"]
)

### Target Variable Considerations

The target variable, total deaths per million population, exhibited substantial right-skewness during exploratory analysis. To evaluate whether transformation improves model performance, both the original and log-transformed outcome variables will be considered during model development.